In [1]:
import os
from pathlib import Path
REPO = Path.cwd().resolve().parents[1]          # docs/notebooks -> repo root
GOLDEN = Path(os.environ.get("CAL_DISP_GOLDEN_DIR", REPO / "test_golden"))
SCRATCH = Path(os.environ.get("TMPDIR", REPO / "tmp")) / "cal_disp_notebooks"; SCRATCH.mkdir(parents=True, exist_ok=True)

# Fix 1 — `cal-disp validate` compares the whole product

**Defect.** `src/cal_disp/validate.py` (gamma-release, `_compare_group` / `_compare_array`,
lines 163–276) compared only the filename fields, dimension names, shapes, the NaN pattern
and the data values of the `main`/`auxiliary` groups. It never looked at attributes
(`units`, `long_name`, `grid_mapping`, `_FillValue`), the CRS WKT, dtypes, the
`identification`/`metadata` values or the browse PNG; a missing group was swallowed
(`except (KeyError, OSError): pass`, line 101); `tolerance` was used as both `rtol` and
`atol` (line 264); pixels where the reference is exactly 0 were skipped (line 260), which
also skipped the millions of clipped `calibration_std` zeros; and the CLI default (1e-6)
differed from the function default (1e-4).

**Fix.** `compare_cal_products` now compares, group by group, the presence of every group,
the variable set, on-disk dtype and shape, all attributes (with a small documented
ignore-list: build versions, `_NCProperties`, `processing_start_datetime`, the embedded
`pge_runconfig`), the CRS as a `rasterio.crs.CRS`, the grid transform, all values (strings
and integers exactly, floats with separate `--rtol`/`--atol`, reference zeros included) and
the browse PNG (≤ 2048 px per side; only a warning when absent). One `DEFAULT_TOLERANCE`
(1e-6) is shared by the function, the CLI and the scripts. Differences are reported grouped
and the CLI exits 1 on any of them.

**Demonstrated here.** The golden product is copied to scratch and mutated one field at a
time with `h5py`; each mutation is run through the *old* comparison (checked out from
`gamma-release` commit `d0901b3` with `git show`) and the *new* one. The table at the end shows which
mutations the old validator missed.

In [2]:
import importlib.util
import logging
import shutil
import subprocess
import time

import h5py
import numpy as np
import pandas as pd

from cal_disp.validate import compare_cal_products

golden_nc = sorted((GOLDEN / "golden_output").glob("OPERA_L4_DISP-CAL-S1_*.nc"))[0]
work_dir = SCRATCH / "08_validation_depth"
work_dir.mkdir(exist_ok=True)
test_nc = work_dir / golden_nc.name

# One working copy (~0.6 GB); mutations below are reverted in place
shutil.copy(golden_nc, test_nc)  # always a fresh copy
test_nc.with_suffix(".png").unlink(missing_ok=True)  # no browse image: only a warning
print(golden_nc.name, f"{golden_nc.stat().st_size / 1e6:.0f} MB")

OPERA_L4_DISP-CAL-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20260930T025407Z.nc 586 MB


## The pre-fix comparison, straight from `gamma-release`

`git show d0901b3:src/cal_disp/validate.py` (`gamma-release` before this fix) is written to scratch and imported as
`validate_old`, so both versions run on exactly the same files.

In [3]:
old_src = subprocess.run(
    ["git", "show", "d0901b3:src/cal_disp/validate.py"],
    cwd=REPO, check=True, capture_output=True, text=True,
).stdout
old_path = work_dir / "validate_old.py"
old_path.write_text(old_src)
spec = importlib.util.spec_from_file_location("validate_old", old_path)
validate_old = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validate_old)

# Both validators log to the same logger; keep only the new one's failure lines
messages: list[str] = []


class _Collect(logging.Handler):
    def emit(self, record):
        if record.levelno >= logging.ERROR:
            messages.append(record.getMessage())


logging.getLogger("cal_disp.validate").addHandler(_Collect())
logging.getLogger("cal_disp.validate").setLevel(logging.ERROR)
logging.getLogger("validate_old").setLevel(logging.ERROR)


def run_both(label: str) -> dict:
    t0 = time.time()
    old_ok = validate_old.compare_cal_products(golden_nc, test_nc, tolerance=1e-6)
    t1 = time.time()
    messages.clear()
    new_ok = compare_cal_products(golden_nc, test_nc, rtol=1e-6, atol=1e-6)
    t2 = time.time()
    # The golden browse PNG is 2506 px wide: reported by the new validator, but not
    # what this notebook is about, so it is listed separately below
    findings = [m for m in messages if m.startswith("  ") and "browse" not in m and ".png" not in m]
    return {
        "mutation": label,
        "old validate": "PASS" if old_ok else "FAIL",
        "new validate": "PASS" if new_ok else "FAIL",
        "new report (first line)": findings[0].strip() if findings else "",
        "old s": round(t1 - t0, 1),
        "new s": round(t2 - t1, 1),
    }

/u/aurora-r0/govorcin/miniconda/miniforge/envs/cal-disp-gamma/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Mutations

Each mutation is applied to the working copy with `h5py`, both validators are run, and the
mutation is reverted. Dropping the `auxiliary` group cannot be reverted, so it goes last.

In [4]:
with h5py.File(test_nc) as f:
    orig_wkt = f["spatial_ref"].attrs["crs_wkt"]
    orig_units = f["calibration"].attrs["units"]
    orig_track = int(f["identification/track_number"][()])
    cal_block = f["calibration"][0, 3000:3100, 4000:4100]
    xs_attrs = dict(f["identification/source_data_x_spacing"].attrs)
    xs_value = f["identification/source_data_x_spacing"][()]
# A finite pixel to perturb
iy, ix = np.argwhere(np.isfinite(cal_block))[0]
iy, ix = int(iy) + 3000, int(ix) + 4000
print("CRS:", orig_wkt[:40], "| units:", orig_units, "| track_number:", orig_track, "| pixel:", (iy, ix))


def set_scalar(f, name, value, dtype, attrs):
    """Recreate a scalar dataset with another dtype (h5py cannot cast in place)."""
    del f[name]
    ds = f.create_dataset(name, data=np.array(value, dtype=dtype))
    for k, v in attrs.items():
        ds.attrs[k] = v


mutations = [
    (
        "CRS WKT: UTM 15N -> 16N",
        lambda f: f["spatial_ref"].attrs.__setitem__(
            "crs_wkt", orig_wkt.replace("zone 15N", "zone 16N").replace('"central_meridian",-93', '"central_meridian",-87')
        ),
        lambda f: f["spatial_ref"].attrs.__setitem__("crs_wkt", orig_wkt),
    ),
    (
        "calibration units: meters -> millimeters",
        lambda f: f["calibration"].attrs.__setitem__("units", "millimeters"),
        lambda f: f["calibration"].attrs.__setitem__("units", orig_units),
    ),
    (
        f"identification/track_number: {orig_track} -> 34",
        lambda f: f["identification/track_number"].__setitem__((), 34),
        lambda f: f["identification/track_number"].__setitem__((), orig_track),
    ),
    (
        f"calibration[{iy}, {ix}] += 1e-5",
        lambda f: f["calibration"].__setitem__((0, iy, ix), f["calibration"][0, iy, ix] + 1e-5),
        lambda f: f["calibration"].__setitem__((0, iy, ix), cal_block[iy - 3000, ix - 4000]),
    ),
    (
        "identification/source_data_x_spacing dtype: float64 -> float32",
        lambda f: set_scalar(f, "identification/source_data_x_spacing", xs_value, np.float32, xs_attrs),
        lambda f: set_scalar(f, "identification/source_data_x_spacing", xs_value, np.float64, xs_attrs),
    ),
    (
        "drop the auxiliary group",
        lambda f: f.__delitem__("auxiliary"),
        None,
    ),
]

rows = [run_both("no mutation (identical copy)")]
for label, apply, revert in mutations:
    with h5py.File(test_nc, "a") as f:
        apply(f)
    rows.append(run_both(label))
    if revert is not None:
        with h5py.File(test_nc, "a") as f:
            revert(f)
    print(f"{rows[-1]['mutation']:60s} old={rows[-1]['old validate']}  new={rows[-1]['new validate']}")

CRS: PROJCS["WGS 84 / UTM zone 15N",GEOGCS["W | units: meters | track_number: 0 | pixel: (3000, 4000)


CRS WKT: UTM 15N -> 16N                                      old=PASS  new=FAIL


calibration units: meters -> millimeters                     old=PASS  new=FAIL


identification/track_number: 0 -> 34                         old=PASS  new=FAIL


Data mismatch for 'calibration':


...

  Tolerance: 1e-06


calibration[3000, 4000] += 1e-5                              old=FAIL  new=FAIL


identification/source_data_x_spacing dtype: float64 -> float32 old=PASS  new=FAIL


auxiliary group mismatch: reference=True, test=False


drop the auxiliary group                                     old=FAIL  new=FAIL


## Result: mutation × {old, new}

In [5]:
pd.set_option("display.max_colwidth", 120)
table = pd.DataFrame(rows).set_index("mutation")
table

,old validate,new validate,new report (first line),old s,new s
mutation,,,,,
no mutation (identical copy),PASS,PASS,,36.4,29.1
CRS WKT: UTM 15N -> 16N,PASS,FAIL,"spatial_ref: CRS EPSG:32616, expected EPSG:32615",151.4,57.7
calibration units: meters -> millimeters,PASS,FAIL,"calibration attribute 'units': 'millimeters', expected 'meters'",159.0,50.7
identification/track_number: 0 -> 34,PASS,FAIL,"track_number: 34, expected 0",44.9,96.9
"calibration[3000, 4000] += 1e-5",FAIL,FAIL,"calibration: 1 of 48396065 values beyond tolerance (max |diff|=1e-05, mean |diff|=2.07e-13, rtol=1e-06, atol=1e-06);...",44.4,152.1
identification/source_data_x_spacing dtype: float64 -> float32,PASS,FAIL,"source_data_x_spacing: dtype float32, expected float64",30.2,114.2
drop the auxiliary group,FAIL,FAIL,group missing from the test product,38.8,7.8


In [6]:
missed = table[(table["old validate"] == "PASS") & (table["new validate"] == "FAIL")]
print(f"{len(missed)} of {len(mutations)} mutations passed the old validator and are caught now:")
for name in missed.index:
    print("  -", name)
assert table.loc["no mutation (identical copy)", "new validate"] == "PASS"
assert (table.drop("no mutation (identical copy)")["new validate"] == "FAIL").all()

4 of 6 mutations passed the old validator and are caught now:
  - CRS WKT: UTM 15N -> 16N
  - calibration units: meters -> millimeters
  - identification/track_number: 0 -> 34
  - identification/source_data_x_spacing dtype: float64 -> float32


## Browse image

The new validator also checks the PNG next to the test product. The golden PNG is wider than
the 2048 px product-spec limit, which the old validator never looked at (the browse image is
fixed in a separate stream).

In [7]:
from PIL import Image

golden_png = golden_nc.with_suffix(".png")
with Image.open(golden_png) as img:
    print(golden_png.name, img.size)
messages.clear()
ok = compare_cal_products(golden_nc, golden_nc, group="main")
print("golden vs itself:", "PASS" if ok else "FAIL")
print("\n".join(m for m in messages if ".png" in m))

OPERA_L4_DISP-CAL-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20260930T025407Z.png (2506, 2048)


golden vs itself: FAIL
  OPERA_L4_DISP-CAL-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20260930T025407Z.png is 2506x2048 px; each side must be <= 2048 px
